# 13.01 · Quantization

**Chapter:** Serving & Deployment · **Milestone:** M5

## Goals
- [ ] Do the memory math: why a 8B model needs ~16 GB in bf16 but ~5 GB in 4-bit
- [ ] Implement absmax quantization (per-tensor, per-channel, group-wise) from scratch and measure what each costs in perplexity
- [ ] Know the formats: GGUF (llama.cpp/Ollama), MLX, AWQ, GPTQ, bitsandbytes, FP8/MXFP4/NVFP4 — and which runs where
- [ ] Quantize one model three ways and compare **size, speed and task score**

## Resources
- [HF – Quantization overview](https://huggingface.co/docs/transformers/main/en/quantization/overview) — free
- [Maarten Grootendorst – A Visual Guide to Quantization](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-quantization) — free
- [llama.cpp – quantize README (k-quants, imatrix)](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md) · [MLX-LM quantization](https://github.com/ml-explore/mlx-lm) — free
- Papers: [LLM.int8()](https://arxiv.org/abs/2208.07339), [GPTQ](https://arxiv.org/abs/2210.17323), [AWQ](https://arxiv.org/abs/2306.00978), [QLoRA/NF4](https://arxiv.org/abs/2305.14314) — free

Serving cost is dominated by **memory**: weights must fit in (GPU/unified) memory, and every generated token reads all of them once.
Decoding is memory-bandwidth bound, so halving the bytes per weight roughly doubles tokens/sec *and* halves the hardware you need.

In [ ]:
import json
import subprocess
import time

import numpy as np
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer

from aieng import DATA, rag

device = "mps" if torch.backends.mps.is_available() else "cpu"

## 1. The memory math
bytes ≈ parameters × bits / 8, plus the KV cache (13.02) and some overhead. Rule of thumb for chat: weights + ~20%.

In [ ]:
for name, params in [("SmolLM2-135M", 0.135e9), ("Qwen3-1.7B", 1.7e9), ("Llama-3.1-8B", 8e9), ("Llama-3.3-70B", 70e9)]:
    row = {f"{label}": f"{params * bits / 8 / 1e9:6.2f} GB" for label, bits in [("fp32", 32), ("bf16", 16), ("int8", 8), ("int4", 4.5)]}
    print(f"{name:14}", row)
print("(int4 counted as 4.5 bits: group-wise scales add ~0.5 bit/weight)")

On a 24 GB Mac: a 70B model is out of reach even at 4-bit, an 8B model fits comfortably at 4-bit or 8-bit, a 1.7B model fits at any precision.

## 2. Quantization from scratch
**Absmax (symmetric) quantization** to `b` bits: scale `s = max|w| / (2^(b-1) - 1)`, store `q = round(w / s)` as an integer, recover `w ≈ q · s`.
The catch is **outliers**: one big weight stretches `s` and crushes every small weight into the same few integer levels.
So we use one scale per *row* (per-channel) or per *group* of 32–128 weights (group-wise; what GGUF, MLX, AWQ and GPTQ all do).

In [ ]:
def fake_quant(w, bits, group=None, per_channel=False):
    """Quantize then dequantize, so we can measure the error in a normal float model."""
    qmax = 2 ** (bits - 1) - 1
    shape = w.shape
    if group:
        w = w.reshape(-1, group)
    if group or per_channel:
        s = w.abs().amax(dim=1, keepdim=True) / qmax
    else:
        s = w.abs().max() / qmax
    s = s.clamp(min=1e-8)
    return ((w / s).round().clamp(-qmax - 1, qmax) * s).reshape(shape)


torch.manual_seed(0)
w = torch.randn(256, 512) * 0.02
w[3, 7] = 1.0  # a single outlier, common in real LLM weights
for label, kw in [("int8 per-tensor", dict(bits=8)), ("int8 per-channel", dict(bits=8, per_channel=True)),
                  ("int4 per-tensor", dict(bits=4)), ("int4 per-channel", dict(bits=4, per_channel=True)), ("int4 group-32", dict(bits=4, group=32))]:
    err = (fake_quant(w, **kw) - w).pow(2).mean().sqrt() / w.std()
    print(f"{label:18} relative RMS error {err:.3f}")

Per-tensor int4 with one outlier: almost every weight rounds to 0 (relative error ≈ 1 means "lost the signal"). Smaller groups confine the damage.

Now the real test: fake-quantize every linear layer of SmolLM2-135M and measure **perplexity** on Wikipedia passages
(perplexity = exp(cross-entropy); lower is better). Embeddings / LM head stay in full precision, as most real schemes do.

In [ ]:
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
text = "\n\n".join(rag.corpus()[:1500])
ids = tok(text, return_tensors="pt").input_ids[0][: 32 * 512].view(32, 512)  # 32 windows of 512 tokens


@torch.no_grad()
def perplexity(model):
    losses = [model(x[None].to(device), labels=x[None].to(device)).loss.item() for x in ids]
    return float(np.exp(np.mean(losses)))


def quantized_model(**kw):
    m = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)
    if kw:
        for mod in m.model.layers.modules():
            if isinstance(mod, nn.Linear):
                mod.weight.data = fake_quant(mod.weight.data, **kw)
    return m.to(device).eval()


results = {}
for label, kw in [("fp32", {}), ("int8 per-tensor", dict(bits=8)), ("int8 per-channel", dict(bits=8, per_channel=True)),
                  ("int4 per-channel", dict(bits=4, per_channel=True)), ("int4 group-128", dict(bits=4, group=128)),
                  ("int4 group-32", dict(bits=4, group=32)), ("int3 group-32", dict(bits=3, group=32))]:
    results[label] = perplexity(quantized_model(**kw))
    print(f"{label:18} perplexity {results[label]:7.2f}")

Typical result: int8 is free, int4 per-channel hurts noticeably, group-wise int4 recovers most of it, 3-bit starts to break.
Tiny models are the *most* sensitive (every weight matters); 8B+ models lose much less at 4-bit. Always measure on your task.

Smarter methods beat round-to-nearest at the same bits:
- **GPTQ**: quantize column by column, adjusting the remaining weights to compensate the error (uses a little calibration data).
- **AWQ**: find the ~1% of weight channels that see large *activations* and scale them up before quantizing so they keep precision.
- **llama.cpp k-quants** (`Q4_K_M`, `Q5_K_M`, …): nested group scales; `imatrix` uses calibration data to weight important weights.
- **NF4** (bitsandbytes, QLoRA): 4-bit levels placed at normal-distribution quantiles instead of evenly.
- **FP8 / MXFP4 / NVFP4**: low-precision *floats* with hardware support on recent NVIDIA GPUs (H100/B200); e.g. gpt-oss ships in MXFP4.

| Format | Runs on | Typical use |
|---|---|---|
| GGUF (`Q4_K_M`, `Q8_0`) | CPU, Apple, CUDA (llama.cpp, Ollama, LM Studio) | local + edge |
| MLX (4/8-bit group-wise) | Apple Silicon only | Mac local |
| AWQ / GPTQ | CUDA (vLLM, SGLang, TGI) | GPU serving |
| bitsandbytes 8/4-bit | CUDA | QLoRA training, quick experiments |
| FP8 / NVFP4 / MXFP4 | Hopper / Blackwell GPUs | high-throughput serving |

## 3. MLX: quantize and measure speed on Apple Silicon
`mlx_lm.convert` downloads a HF model and writes a quantized MLX copy (group size 64 by default).

In [ ]:
from mlx_lm import convert, load, stream_generate

HF_MODEL = "Qwen/Qwen3-0.6B"
MLX = DATA / "mlx-quant"
for bits in (None, 8, 4):
    out = MLX / f"qwen3-0.6b-{bits or 'bf16'}"
    if not out.exists():
        convert(HF_MODEL, mlx_path=str(out), quantize=bits is not None, **({"q_bits": bits} if bits else {}), dtype="bfloat16")

import mlx.core as mx
import mlx.nn as mnn


def mlx_ppl(model, tokenizer):
    x = mx.array(tokenizer.encode(text)[: 16 * 512]).reshape(16, 512)
    logits = model(x[:, :-1])
    return float(mx.exp(mnn.losses.cross_entropy(logits, x[:, 1:]).mean()))


prompt = "Explain in two sentences why quantization makes LLM inference faster."
for path in [MLX / f"qwen3-0.6b-{b}" for b in ("bf16", 8, 4)]:
    model = tokenizer = None; mx.clear_cache(); mx.reset_peak_memory()  # free the previous model first
    model, tokenizer = load(str(path))
    msgs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, enable_thinking=False)
    for r in stream_generate(model, tokenizer, msgs, max_tokens=100):
        pass
    size = sum(f.stat().st_size for f in path.glob("*.safetensors")) / 1e9
    print(f"{path.name:18} {size:.2f} GB | {r.generation_tps:5.0f} tok/s | peak mem {r.peak_memory:.2f} GB | ppl {mlx_ppl(model, tokenizer):6.2f}")

Same model, three sizes: speed scales roughly with 1 / bytes per weight because decoding just streams the weights through memory.

## 4. Ollama: quantize the fine-tuned model three ways and compare on its own task
Back to M6's `smollm-payments` (12.06). Ollama quantizes on import (`--quantize int8 | int4`; older versions and llama.cpp use GGUF names
like `q8_0`, `q4_K_M`). We score each variant on the held-out templates.

In [ ]:
import sys
sys.path.insert(0, "../12-fine-tuning")
from task import dataset, score

from aieng.llm import client

MERGED = DATA / "ft" / "smollm-payments-merged"
assert (MERGED / "Modelfile").exists(), "Run 12.06 first (it writes the merged model + Modelfile)."
oc, _ = client("ollama")
test_ood = [e for e in dataset(800, seed=3) if e["text"].startswith(("Netflix charged me", "me cobraron"))][:50]

for quant in (None, "int8", "int4"):
    name = f"smollm-payments-{quant or 'full'}"
    cmd = ["ollama", "create", name, "-f", str(MERGED / "Modelfile")] + (["--quantize", quant] if quant else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-300:]
    oc.chat.completions.create(model=name, messages=[{"role": "user", "content": "warm up"}], max_tokens=1)  # load into memory
    start, scores = time.perf_counter(), []
    for e in test_ood:
        out = oc.chat.completions.create(model=name, messages=[{"role": "user", "content": e["text"]}], temperature=0, max_tokens=64)
        scores.append(score(out.choices[0].message.content, e["target"])["exact"])
    ms = (time.perf_counter() - start) / len(test_ood) * 1000
    size = next(l.split()[2] + " " + l.split()[3] for l in subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout.splitlines() if l.startswith(name))
    print(f"{name:24} {size:>8} | exact {np.mean(scores):4.0%} | {ms:4.0f} ms/request")

Read the table like an engineer: if int4 keeps the task score, ship int4 (smaller, cheaper, faster). If it drops, int8 is the safe default.
For a 135M model latency is dominated by overhead, not weights, so the speed gap is small; on a 8B model it is ~2×.
int4 is not half of int8 here: the 49k-token embedding table is a big share of a tiny model and is kept at higher precision.

## Takeaways
- 8-bit is near-lossless almost always; 4-bit group-wise is the sweet spot for local; below 4 bits needs smarter methods and evals.
- Pick the format by *where it runs*: GGUF/Ollama anywhere, MLX on Mac, AWQ/GPTQ/FP8 on NVIDIA servers.
- Quantization also applies to the **KV cache** (13.02) and to activations (W8A8, FP8) for more throughput on big GPUs.

## Exercises
1. Add `int3 group-128` and `int2 group-32` to §2. Where does the model collapse? Plot perplexity vs bits-per-weight.
2. Install llama.cpp (`brew install llama.cpp`), convert `smollm-payments-merged` with `convert_hf_to_gguf.py`, try `Q8_0`, `Q4_K_M`, `Q2_K`,
   serve with `llama-server` and score it with the same loop as §4.
3. Quantize `Qwen/Qwen3-1.7B` with MLX at 4 and 8 bits; compare against `mlx-community/Qwen3-1.7B-4bit` (already in your cache from 12.03).

In [ ]:
# your code here